# Algorithm correctness narrative: Bernstein-Vazirani & Grover success probabilities

**Pack:** `samples/research/` (#190) — literate notebook, paired with
[`algorithm_correctness_narrative_smoke.py`](./algorithm_correctness_narrative_smoke.py).

**Question:** for two canonical oracle algorithms, does the closed-form
probability derived on paper actually match what Qiskit Aer samples from the
`quonc`-compiled circuit?

**Linked canonical sources:**
[`test/verify/bernstein_vazirani.qn`](../../test/verify/bernstein_vazirani.qn),
[`test/verify/grover.qn`](../../test/verify/grover.qn) — both already carry
`ci: smoke` catalog rows in this pack (`research/bernstein-vazirani-oracle`,
`research/grover-n4-marked-11`) that typecheck them via `quonc`; this
notebook narrates the *same* files, it does not fork copies.

**Not a `test/verify/` re-implementation:** `test/verify/bernstein_vazirani.py`
and `test/verify/grover.py` already gate these fixtures' compiled semantics
in `just ci-rust`. This notebook instead derives *why* the expected
probability is what it is, then treats the existing Aer bridge
([`python/quon_aer.py`](../../python/quon_aer.py)) as an oracle to confirm
the derivation — a different question than "did the compiler break
something."

## Setup

```bash
cargo build --release -p quonc
export QUONC=$PWD/target/release/quonc
pip install -r python/requirements.txt   # qiskit, qiskit-aer, qiskit-qasm3-import
```

As with every notebook in this pack, code cells below show real commands
with no fabricated outputs — see
[`repro_appendix_template.md`](./repro_appendix_template.md). The numbers in
the prose come from running
[`algorithm_correctness_narrative_smoke.py`](./algorithm_correctness_narrative_smoke.py)
at SHOTS=4096, SEED=190, against commit
`d26723141d5799a0e8107915b16c55ef3b9fa6f3`.

## Part 1: Bernstein-Vazirani — single-shot exact recovery

`bernstein_vazirani.qn` fixes the secret $s = 110$ over $n=3$ query qubits
plus one ancilla (see its header comment for the oracle construction:
$\mathrm{CNOT}(q_i, \mathrm{anc})$ for each $i$ with $s_i = 1$).

**Derivation.** Bernstein & Vazirani (1997) show the BV oracle circuit
($H^{\otimes n} \to U_f \to H^{\otimes n}$) maps
$|0\rangle^{\otimes n} \mapsto |s\rangle$ *exactly*, with no residual
amplitude on any other basis state — one oracle query recovers all $n$ bits
of $s$ deterministically. So the theoretical single-shot recovery
probability is exactly $1.0$, for any $s$, not just $s=110$.

In [ ]:
$QUONC --emit-qasm -q test/verify/bernstein_vazirani.qn | \
  python python/quon_aer.py --shots 4096 --seed 190
# every one of 4096 shots reports (c0,c1,c2) = (1,1,0) = s

**Confirmed:** sampling 4096 shots at a fixed seed, every shot's
`(c0, c1, c2)` equals the secret `(1, 1, 0)` — a sampled recovery rate of
exactly `1.0`, matching the theoretical prediction with no tolerance needed
(there is no distribution to approximate; BV's oracle is deterministic up to
gate-level noise, and Aer's simulation here is noiseless).

## Part 2: Grover — exact success probability at N=4, M=1

`grover.qn` runs Grover's algorithm at $n=2$ ($N=2^n=4$ states), one marked
item $|11\rangle$, for $r=1$ iteration (see its header comment).

**Derivation.** Grover's amplitude-amplification recurrence gives success
probability $P(r) = \sin^2\big((2r+1)\theta\big)$ where
$\theta = \arcsin(1/\sqrt{N})$ (Grover 1996; Nielsen & Chuang §6.1). For
$N=4$: $\theta = \arcsin(1/2) = \pi/6$. At $r=1$:
$$P(1) = \sin^2\!\big(3 \cdot \tfrac{\pi}{6}\big) = \sin^2\!\big(\tfrac{\pi}{2}\big) = 1.0$$
This is Grover's *exact* special case — $N=4, M=1$ is one of the few
configurations where a single iteration recovers the marked item with
probability exactly 1, not just approximately (contrast with larger $N$,
where $P(r)$ oscillates through non-unit maxima).

In [ ]:
$QUONC --emit-qasm -q test/verify/grover.qn | \
  python python/quon_aer.py --shots 4096 --seed 190
# every one of 4096 shots reports the marked state |11>

**Confirmed:** all 4096 sampled shots land on `11`, giving a Hellinger
fidelity of `1.0000` against the point-mass expected distribution
`{"11": 1.0}` — matching the theoretical $P(1) = 1.0$ exactly.

## Why this pair, together

BV and Grover sit at opposite ends of the same question — "how many oracle
queries does it take to learn something classically hard?" — with opposite
*exactness* stories: BV is deterministically exact for *any* problem size
$n$ (one query, always), while Grover is only exactly deterministic at this
one special ratio $M/N = 1/4$; for other $N$, $\lceil \pi\sqrt{N}/4 \rceil$
iterations gets you *close* to, but not exactly, probability 1. Both
notebook cells above sample real `quonc`-compiled circuits and confirm both
derivations hold — the theory isn't just a comment in the `.qn` source, it's
checked against the compiled artifact.

## Repro appendix

- **Quon commit:** `d26723141d5799a0e8107915b16c55ef3b9fa6f3` (branch
  `samples/190-research-notebooks`)
- **`quonc` version:** `quonc 0.1.0`
- **Build:** `cargo build --release -p quonc`
- **Python:** `Python 3.9+`, deps from
  [`python/requirements.txt`](../../python/requirements.txt)
  (`pip install -r python/requirements.txt`; needs `qiskit`, `qiskit-aer`,
  `qiskit-qasm3-import` — the smoke twin skips gracefully if not installed)
- **Smoke twin:** `python samples/research/algorithm_correctness_narrative_smoke.py`
  (SHOTS=4096, SEED=190)
- **Linked canonical sources:** `test/verify/bernstein_vazirani.qn`
  (`research/bernstein-vazirani-oracle`), `test/verify/grover.qn`
  (`research/grover-n4-marked-11`)